# Training Notebook

In [1]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack import get_no_mod, LWEDataset, get_filename_from_params
from ml_attack.utils import get_continuous_reduction_default_params, get_default_params, get_percentage_true_b, get_train_default_params, cmod, mod_mult

import numpy as np

from scipy.linalg import circulant
from ml_attack.lwe import neg_circ

from ml_attack.continuous_reduction import ContinuousReduction
from concurrent.futures import ProcessPoolExecutor

from itertools import product

np.set_printoptions(linewidth=np.inf)

## Dataset creation

Training debug:

In [3]:
params = get_default_params()
params.update(get_continuous_reduction_default_params())
params.update(get_train_default_params())
params.update({
    'n': 32,
    'q': 3329,
    'k': 2,
    'eta': 2,
    'secret_type': 'cbd',
    'error_type': 'cbd',

    'num_gen': 4,
    'seed': 42,

    'num_matrices': 6,
    'reduction_max_size': 10,
    'float_type': 'd',
    'matrix_config': 'salsa',
    'interleaved_steps': 10,
    'reduction_samples': 0.5,
    'reduction_resampling': False,
    'warmup_steps': 2,
    'bkz_block_sizes': "20:40:10",
    
    'penalty': 4,
    'verbose': True,
    'continuous_reduction': True,

    "train_percentages": [0.1, 0.3, 0.6, 1]
})

filename = get_filename_from_params(params)

reload = False
if os.path.exists(filename) and reload:
    print(f"Loading dataset from {filename}")
    dataset = LWEDataset.load_reduced(filename)
    params = dataset.params
    dataset.approximate_b()
else:
    print(f"Generating dataset and saving to {filename}")
    dataset = LWEDataset(params)
    dataset.initialize()
    dataset.attack(
        stop_strategy="tour",
        stop_after=4,
        attack_strategy="no",
        save_at_the_end=True
    )

Generating dataset and saving to ./data_n_32_k_2_s_cbd_62d67.pkl
Attacking 8 matrices using 8 threads.
- Algo: flatter | Updated 10/96 | Mean std_B: 1920.57
- Algo: flatter | Updated 10/96 | Mean std_B: 1897.06- Algo: flatter | Updated 10/96 | Mean std_B: 1849.26

- Algo: flatter | Updated 10/96 | Mean std_B: 1925.68
- Algo: flatter | Updated 10/96 | Mean std_B: 1867.42
- Algo: flatter | Updated 10/96 | Mean std_B: 1861.35- Algo: flatter | Updated 10/96 | Mean std_B: 1921.82

- Algo: flatter | Updated 10/96 | Mean std_B: 1895.82
Tour 1 | Time: 6.65s | Mean std_B: 1617.96 | Reduction Factor: 0.1986 | Prob: 0.6964
- Algo: flatter | Updated 9/96 | Mean std_B: 1834.84
- Algo: flatter | Updated 4/96 | Mean std_B: 1805.95
- Algo: flatter | Updated 7/96 | Mean std_B: 1836.39
- Algo: flatter | Updated 8/96 | Mean std_B: 1725.83
- Algo: flatter | Updated 8/96 | Mean std_B: 1799.67
- Algo: flatter | Updated 8/96 | Mean std_B: 1773.32
- Algo: flatter | Updated 8/96 | Mean std_B: 1764.99
- Algo: f

In [4]:
dataset.approximate_b()
get_percentage_true_b(dataset, verbose=True)

True B is the best candidate: 2199 / 2560 (85.90%)


np.float64(0.858984375)

In [5]:
num_gen = dataset.params['num_gen']
n = dataset.params['n']
k = dataset.params['k']
q = dataset.params['q']

In [6]:
dataset.A.shape

(64, 16)

In [7]:
dataset.A

array([[  963.,  -633.,  1239., ...,   923.,    34., -1299.],
       [  760.,   963.,  -633., ...,   310.,   923.,    34.],
       [-1069.,   760.,   963., ...,  1332.,   310.,   923.],
       ...,
       [ 1598., -1303., -1586., ...,  -205.,    70.,  -888.],
       [ 1009.,  1598., -1303., ...,  -262.,  -205.,    70.],
       [ -869.,  1009.,  1598., ...,  1226.,  -262.,  -205.]])

In [18]:
A_to_reduce = np.stack([dataset.A[ind] for ind in dataset.indices])
A_to_reduce[0]

array([[  963.,  -633.,  1239.,  1368.,  -142., -1076.,  1069.,  -760.,  -929., -1293., -1208.,  1332.,   310.,   923.,    34., -1299.],
       [  760.,   963.,  -633.,  1239.,  1368.,  -142., -1076.,  1069.,  1299.,  -929., -1293., -1208.,  1332.,   310.,   923.,    34.],
       [-1069.,   760.,   963.,  -633.,  1239.,  1368.,  -142., -1076.,   -34.,  1299.,  -929., -1293., -1208.,  1332.,   310.,   923.],
       [ 1076., -1069.,   760.,   963.,  -633.,  1239.,  1368.,  -142.,  -923.,   -34.,  1299.,  -929., -1293., -1208.,  1332.,   310.],
       [  142.,  1076., -1069.,   760.,   963.,  -633.,  1239.,  1368.,  -310.,  -923.,   -34.,  1299.,  -929., -1293., -1208.,  1332.],
       [-1368.,   142.,  1076., -1069.,   760.,   963.,  -633.,  1239., -1332.,  -310.,  -923.,   -34.,  1299.,  -929., -1293., -1208.],
       [-1239., -1368.,   142.,  1076., -1069.,   760.,   963.,  -633.,  1208., -1332.,  -310.,  -923.,   -34.,  1299.,  -929., -1293.],
       [  633., -1239., -1368.,   142.,  

In [17]:
dataset.R[0][0]

array([[  8.,  43.,   6.,  42., -37.,   2.,  18.,  23.],
       [-42.,   8.,  43.,   6., -23., -37.,   2.,  18.],
       [ -6., -42.,   8.,  43., -18., -23., -37.,   2.],
       [-43.,  -6., -42.,   8.,  -2., -18., -23., -37.]])

In [13]:
mod_mult(dataset.R[0][0][0], A_to_reduce[0], q)

array([ 178., -101., -102.,   33.,  -56.,   50.,  -65.,   71., -165.,   72.,  181., -147.,   16.,  -53., -104., -159.])

In [10]:
parts = np.split(dataset.R[0][0], 1)
parts = np.hstack([neg_circ(part).T for part in parts])
parts

array([[ 31.,  41.,  -8., ...,  72.,   3.,   0.],
       [ -0.,  31.,  41., ...,  18.,  72.,   3.],
       [ -3.,  -0.,  31., ...,  16.,  18.,  72.],
       ...,
       [-31., -28.,  56., ...,  31.,  41.,  -8.],
       [  8., -31., -28., ...,  -0.,  31.,  41.],
       [-41.,   8., -31., ...,  -3.,  -0.,  31.]])

From paper "Enhancing MLWE"

In [54]:
import numpy as np
from fpylll import IntegerMatrix

def build_full_basis(A, q):
    m, d = A.shape  # m samples, d = k*n
    I_m = np.eye(m, dtype=int)
    I_d = np.eye(d, dtype=int)
    
    # Block matrix
    top = np.hstack([I_m, np.zeros((m, d), dtype=int)])
    bottom = np.hstack([A.T % q, q * I_d])
    B = np.vstack([top, bottom])
    
    return B

def project_to_V(B, n, h, g, k):
    rows, cols = B.shape
    Pi = np.eye(rows, dtype=int)
    for j in range(h*n + g, (h+1)*n):
        Pi[j, j] = 0
    return Pi @ B

from fpylll import LLL

def construct_Lprime_basis(A, q, n, h, g, k):
    # Step 1: Build full basis
    B = build_full_basis(A, q)
    B = IntegerMatrix.from_matrix(B.tolist())
    
    # Step 2: Dual basis (inverse transpose)
    B_np = np.array(B, dtype=int)
    B_dual = np.linalg.inv(B_np).T  # rational entries!
    
    # Step 3: Projection
    PiB = project_to_V(B_dual, n, h, g, k)
    
    # Step 4: Apply LLL (MLLL if available)
    D = IntegerMatrix.from_matrix(PiB.astype(int).tolist())
    LLL.reduction(D)
    
    # Step 5: Delete zero rows
    D_reduced = [row for row in D if any(x != 0 for x in row)]
    D_reduced = IntegerMatrix.from_matrix(D_reduced)
    
    # Step 6: Back to primal
    D_np = np.array(D_reduced, dtype=int)
    B_prime = np.linalg.inv(D_np).T
    
    return B_prime



In [55]:
import numpy as np
from fractions import Fraction

# Parameters
n = 4
k = 2
q = 251
m = 7                # not divisible by n
h = m // n           # =1
g = m - h*n          # =2

dim_u_full = (h+1)*n # 8
dim_v = k*n          # 4
ambient_dim = dim_u_full + dim_v  # 12

print("Parameters:", dict(n=n, k=k, q=q, m=m, h=h, g=g,
                          ambient_dim=ambient_dim, rank_Lp=m+dim_v))

# Random A_full of shape ( (h+1)n x kn )
rng = np.random.default_rng(42)
A_full = rng.integers(low=0, high=q, size=(dim_u_full, dim_v))
A_used = A_full[:m, :]

print("\nA_full (8x4):\n", A_full)
print("\nA_used (first 6 rows):\n", A_used)

# --- Build ambient lattice basis B_full ---
I_u = np.eye(dim_u_full, dtype=int)
I_v = np.eye(dim_v, dtype=int)

top = np.hstack([I_u, np.zeros((dim_u_full, dim_v), dtype=int)])
bottom = np.hstack([A_full.T % q, q*I_v])
B_full = np.vstack([top, bottom])

print("\nB_full (12x12) lattice basis:\n", B_full)

# --- Build q * B^{-T} (integer-scaled dual basis) ---
B_invT_scaled = np.zeros((ambient_dim, ambient_dim), dtype=int)
# top-left: qI
for i in range(dim_u_full):
    B_invT_scaled[i, i] = q
# top-right: -A_full
B_invT_scaled[:dim_u_full, dim_u_full:] = -A_full
# bottom-right: I
for j in range(dim_v):
    B_invT_scaled[dim_u_full+j, dim_u_full+j] = 1

print("\nq * B^{-T} (12x12):\n", B_invT_scaled)

# --- Projection Π: zero out u-rows hn+g .. (h+1)n-1 ---
Pi = np.eye(ambient_dim, dtype=int)
for r in range(h*n + g, (h+1)*n):
    Pi[r, r] = 0
print("\nProjection Pi:\n", Pi)

# Apply projection
D_scaled = Pi @ B_invT_scaled
print("\nD_scaled = Pi * (q * B^{-T}):\n", D_scaled)

# --- Reduced square basis B_red for L' (size (m+kn)x(m+kn)) ---
I_m = np.eye(m, dtype=int)
top_red = np.hstack([I_m, np.zeros((m, dim_v), dtype=int)])
bottom_red = np.hstack([A_used.T % q, q*I_v])
B_red = np.vstack([top_red, bottom_red])

print("\nB_red (10x10) — the practical square basis for L':\n", B_red)



Parameters: {'n': 4, 'k': 2, 'q': 251, 'm': 7, 'h': 1, 'g': 3, 'ambient_dim': 16, 'rank_Lp': 15}

A_full (8x4):
 [[ 22 194 164 110 108 215  21 175]
 [ 50  23 132 244 184 191 180 197]
 [128  32 210 113 125  93  45 232]
 [196 161 101 206 136 111 113  57]
 [ 23 139 222  16 215 207  69 158]
 [ 41 190 175  88  17 243 111 224]
 [170 195 190  48  91 117 124  10]
 [137  38 186 171 231 186  92 242]]

A_used (first 6 rows):
 [[ 22 194 164 110 108 215  21 175]
 [ 50  23 132 244 184 191 180 197]
 [128  32 210 113 125  93  45 232]
 [196 161 101 206 136 111 113  57]
 [ 23 139 222  16 215 207  69 158]
 [ 41 190 175  88  17 243 111 224]
 [170 195 190  48  91 117 124  10]]

B_full (12x12) lattice basis:
 [[  1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   1   0   0   0  